In [ ]:
# ------------------------------------------------------------------------------
# Cell 1: Install Dependencies
# ------------------------------------------------------------------------------
!pip install fastapi uvicorn nest_asyncio requests huggingface_hub python-multipart
!pip install av comfyui-frontend-package comfyui-workflow-templates spandrel torchsde trampoline sentencepiece protobuf --no-deps

In [ ]:
# ------------------------------------------------------------------------------
# Cell 2: Imports & Configuration
# ------------------------------------------------------------------------------
import os
import sys
import time
import subprocess
import shutil
from pathlib import Path
import nest_asyncio

nest_asyncio.apply()

class CONFIG:
    COMFY_DIR = "/kaggle/working/ComfyUI"
    COMFY_PORT = 8000
    FACADE_PORT = 8001
    OUTPUTS_DIR = "/kaggle/working/comfy_outputs"
    ZROK_TOKEN_PATH = "/kaggle/input/sage-zrok-token/.zrok_api_key"
    ZROK_BINARY = "./zrok"
    ZROK_VERSION = "1.1.3"

Path(CONFIG.OUTPUTS_DIR).mkdir(parents=True, exist_ok=True)
Path(CONFIG.COMFY_DIR).mkdir(parents=True, exist_ok=True)
print("✅ Config ready.")

In [ ]:
# ------------------------------------------------------------------------------
# Cell 3: Setup Z-Image (Text-to-Image)
# ------------------------------------------------------------------------------
import os
import subprocess
import shutil

COMFY_DIR = CONFIG.COMFY_DIR

# 1. Clone ComfyUI
if not os.path.exists(os.path.join(COMFY_DIR, "main.py")):
    print("Cloning ComfyUI...")
    if os.path.exists(COMFY_DIR): shutil.rmtree(COMFY_DIR)
    subprocess.run(f"git clone -q https://github.com/comfyanonymous/ComfyUI {COMFY_DIR}", shell=True, check=True)
    print("✅ ComfyUI cloned.")
else:
    print("✅ ComfyUI already present.")

# 2. Install Requirements
print("Installing ComfyUI requirements...")
subprocess.run(f"pip install -q -r {COMFY_DIR}/requirements.txt", shell=True)
print("✅ Requirements installed.")

# 3. Directory Structure
dirs = {
    "text_encoders": f"{COMFY_DIR}/models/text_encoders",
    "diffusion": f"{COMFY_DIR}/models/diffusion_models",
    "vae": f"{COMFY_DIR}/models/vae"
}
for d in dirs.values(): os.makedirs(d, exist_ok=True)

# 4. Download Helper
def download_quiet(url, directory):
    filename = url.split('/')[-1]
    dest = os.path.join(directory, filename)
    if not os.path.exists(dest):
        print(f"⏳ Downloading {filename}...")
        subprocess.run(f"wget -nv -c -O '{dest}' '{url}'", shell=True, check=True)
        print(f"✅ Downloaded {filename}")
    else:
        print(f"✅ {filename} already exists.")

print("\nDownloading Z-Image model weights...")

# Z-Image model downloads
# Text Encoder: Qwen 3 4B (shared with z_image_turbo)
download_quiet(
    "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors",
    dirs["text_encoders"]
)

# Diffusion Model: Z-Image BF16
download_quiet(
    "https://huggingface.co/Comfy-Org/z_image/resolve/main/split_files/diffusion_models/z_image_bf16.safetensors",
    dirs["diffusion"]
)

# VAE: ae.safetensors (shared with z_image_turbo)
download_quiet(
    "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors",
    dirs["vae"]
)

print("\n✅ All Z-Image models ready.")

In [ ]:
# ------------------------------------------------------------------------------
# Cell 4: Additional Setup
# ------------------------------------------------------------------------------
import os
from pathlib import Path

Path("/kaggle/working/ComfyUI/input").mkdir(parents=True, exist_ok=True)
print("✅ Additional directories ready.")

In [ ]:
# ------------------------------------------------------------------------------
# Cell 5: Create Headless API Script (zimage_api_t2i.py)
# Mirrors Z-Image workflow: UNETLoader -> CLIPLoader (lumina2) -> VAELoader
# -> EmptySD3LatentImage -> CLIPTextEncode x2 -> ModelSamplingAuraFlow
# -> KSampler (res_multistep) -> VAEDecode -> SaveImage
# ------------------------------------------------------------------------------

code_content = '''
import os
import sys
import json
import time
import uuid
import requests
import uvicorn
from pathlib import Path
from typing import Optional
from fastapi import FastAPI, Form, BackgroundTasks
from fastapi.responses import StreamingResponse, JSONResponse

# --- CONFIG ---
class ZIMAGE_CONFIG:
    COMFY_PORT = 8000
    FACADE_PORT = 8001
    OUTPUTS_DIR = "/kaggle/working/comfy_outputs"

    # Z-Image model filenames
    UNET_NAME  = "z_image_bf16.safetensors"
    CLIP_NAME  = "qwen_3_4b.safetensors"
    VAE_NAME   = "ae.safetensors"

    # Z-Image recommended defaults (original: 30-50 steps, cfg 3-5)
    DEFAULT_STEPS = 30
    DEFAULT_CFG   = 4.0
    DEFAULT_W     = 1024
    DEFAULT_H     = 1024

Path(ZIMAGE_CONFIG.OUTPUTS_DIR).mkdir(parents=True, exist_ok=True)

# --- COMFY HELPERS ---
def comfy_post(path, json_payload=None):
    try: return requests.post(f"http://localhost:{ZIMAGE_CONFIG.COMFY_PORT}{path}", json=json_payload, timeout=30)
    except Exception as e: print(f"[comfy_post error] {e}"); return None

def comfy_get(path):
    try: return requests.get(f"http://localhost:{ZIMAGE_CONFIG.COMFY_PORT}{path}", timeout=30)
    except Exception as e: print(f"[comfy_get error] {e}"); return None

# --- WORKFLOW BUILDER ---
# Replicates the official Z-Image ComfyUI workflow exactly:
#   UNETLoader -> ModelSamplingAuraFlow (shift=3)
#   CLIPLoader (type=lumina2) -> CLIPTextEncode x2
#   VAELoader -> EmptySD3LatentImage
#   KSampler (res_multistep, simple) -> VAEDecode -> SaveImage
def build_zimage_workflow(prompt, negative_prompt="", width=None, height=None, steps=None, cfg=None, seed=None):
    if seed   is None: seed   = int(uuid.uuid4().int & (2**31 - 1))
    if width  is None: width  = ZIMAGE_CONFIG.DEFAULT_W
    if height is None: height = ZIMAGE_CONFIG.DEFAULT_H
    if steps  is None: steps  = ZIMAGE_CONFIG.DEFAULT_STEPS
    if cfg    is None: cfg    = ZIMAGE_CONFIG.DEFAULT_CFG

    wf = {
        # Step 1: Load Models
        "10": {
            "class_type": "UNETLoader",
            "inputs": { "unet_name": ZIMAGE_CONFIG.UNET_NAME, "weight_dtype": "default" }
        },
        "11": {
            "class_type": "CLIPLoader",
            # Z-Image uses lumina2 type for Qwen encoder (matches official workflow)
            "inputs": { "clip_name": ZIMAGE_CONFIG.CLIP_NAME, "type": "lumina2", "device": "default" }
        },
        "12": {
            "class_type": "VAELoader",
            "inputs": { "vae_name": ZIMAGE_CONFIG.VAE_NAME }
        },

        # ModelSamplingAuraFlow wrapper (shift=3, matches official workflow)
        "13": {
            "class_type": "ModelSamplingAuraFlow",
            "inputs": { "model": ["10", 0], "shift": 3.0 }
        },

        # Step 2: Image size (EmptySD3LatentImage for Z-Image / DiT)
        "3": {
            "class_type": "EmptySD3LatentImage",
            "inputs": { "width": width, "height": height, "batch_size": 1 }
        },

        # Step 3: Prompts
        "6": {
            "class_type": "CLIPTextEncode",
            "inputs": { "text": prompt, "clip": ["11", 0] }
        },
        "7": {
            "class_type": "CLIPTextEncode",
            "inputs": { "text": negative_prompt, "clip": ["11", 0] }
        },

        # KSampler — res_multistep + simple scheduler (matches official workflow)
        "4": {
            "class_type": "KSampler",
            "inputs": {
                "model":        ["13", 0],
                "positive":     ["6",  0],
                "negative":     ["7",  0],
                "latent_image": ["3",  0],
                "seed":         seed,
                "steps":        steps,
                "cfg":          cfg,
                "sampler_name": "res_multistep",
                "scheduler":    "simple",
                "denoise":      1.0
            }
        },

        # Decode & Save
        "8": {
            "class_type": "VAEDecode",
            "inputs": { "samples": ["4", 0], "vae": ["12", 0] }
        },
        "9": {
            "class_type": "SaveImage",
            "inputs": { "filename_prefix": "z-image", "images": ["8", 0] }
        }
    }
    return wf


def process_workflow_bg(workflow_dict, task_id):
    status_file = Path(ZIMAGE_CONFIG.OUTPUTS_DIR) / f"task_{task_id}.json"
    status_file.write_text(json.dumps({"status": "processing"}))

    resp = comfy_post("/prompt", {"prompt": workflow_dict})
    if not resp:
        status_file.write_text(json.dumps({"status": "error", "detail": "ComfyUI unreachable"}))
        return

    try: prompt_id = resp.json().get("prompt_id")
    except:
        status_file.write_text(json.dumps({"status": "error", "detail": "Invalid Comfy response"}))
        return

    # Poll Loop — 900s timeout (15 min, enough for 30-50 steps on Z-Image)
    for _ in range(900):
        hist = comfy_get(f"/history/{prompt_id}")
        if hist and prompt_id in hist.json():
            data = hist.json()[prompt_id]
            if "status" in data and data["status"].get("status_str") == "error":
                status_file.write_text(json.dumps({"status": "error", "detail": "ComfyUI Workflow Error"}))
                return

            files = []
            for out in data.get("outputs", {}).values():
                for img in out.get("images", []):
                    fname = img["filename"]
                    raw = comfy_get(f"/view?filename={fname}&type=output")
                    local_p = Path(ZIMAGE_CONFIG.OUTPUTS_DIR) / f"{prompt_id}_{fname}"
                    local_p.write_bytes(raw.content)
                    files.append(local_p.name)

            status_file.write_text(json.dumps({"status": "completed", "files": files}))
            return
        time.sleep(1)

    status_file.write_text(json.dumps({"status": "error", "detail": "Timeout (900s)"}))


# --- FASTAPI APP ---
app = FastAPI(title="Z-Image T2I API", description="Tongyi Z-Image 6B text-to-image via ComfyUI")

@app.get("/")
def health():
    return {"status": "Z-Image T2I API is Alive", "model": ZIMAGE_CONFIG.UNET_NAME}

@app.post("/generate-async")
async def generate_async(
    background_tasks: BackgroundTasks,
    prompt: str = Form(...),
    negative_prompt: str = Form(""),
    num_inference_steps: int = Form(30),
    height: int = Form(1024),
    width: int = Form(1024),
    guidance_scale: float = Form(4.0),
    seed: int = Form(None)
):
    task_id = uuid.uuid4().hex
    wf = build_zimage_workflow(
        prompt=prompt,
        negative_prompt=negative_prompt,
        width=width,
        height=height,
        steps=num_inference_steps,
        cfg=guidance_scale,
        seed=seed
    )
    background_tasks.add_task(process_workflow_bg, wf, task_id)
    return {"task_id": task_id, "status": "queued"}

@app.get("/status/{task_id}")
def get_status(task_id: str):
    p = Path(ZIMAGE_CONFIG.OUTPUTS_DIR) / f"task_{task_id}.json"
    if not p.exists(): return {"status": "pending"}
    data = json.loads(p.read_text())
    if data.get("status") == "completed" and data.get("files"):
        img_path = Path(ZIMAGE_CONFIG.OUTPUTS_DIR) / data["files"][0]
        if img_path.exists():
            return StreamingResponse(open(img_path, "rb"), media_type="image/png")
    return data

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=ZIMAGE_CONFIG.FACADE_PORT)
'''

with open("zimage_api_t2i.py", "w") as f:
    f.write(code_content)

print("✅ 'zimage_api_t2i.py' created.")

In [ ]:
# ------------------------------------------------------------------------------
# Cell 6: Start System (ComfyUI + Headless API + Zrok Tunnel)
# Identical zrok setup as flux2-klein notebook
# ------------------------------------------------------------------------------
import subprocess
import sys
import time
import os
import re
import glob
import requests

# Config
COMFY_DIR    = "/kaggle/working/ComfyUI"
COMFY_PORT   = 8000
FACADE_PORT  = 8001
ZROK_BINARY  = "./zrok"

# Glob-based token path — works regardless of how Kaggle mounts the dataset
_token_matches = glob.glob("/kaggle/input/**/.zrok_api_key", recursive=True)
ZROK_TOKEN_PATH = _token_matches[0] if _token_matches else None

def get_zrok_token():
    if ZROK_TOKEN_PATH and os.path.isfile(ZROK_TOKEN_PATH):
        with open(ZROK_TOKEN_PATH, "r") as f: return f.read().strip()
    return None

def force_zrok_auth(token):
    if not os.path.exists(ZROK_BINARY):
        print("⬇️ Downloading Zrok v1.1.3...")
        subprocess.run(
            "wget -q https://github.com/openziti/zrok/releases/download/v1.1.3/zrok_1.1.3_linux_amd64.tar.gz -O zrok.tar.gz",
            shell=True
        )
        subprocess.run("tar -xzf zrok.tar.gz && chmod +x zrok", shell=True)
    subprocess.run("chmod +x ./zrok", shell=True)
    subprocess.run(f"{ZROK_BINARY} disable", shell=True, stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
    print("🔄 Authenticating Zrok...")
    res = subprocess.run(
        f'{ZROK_BINARY} enable --headless "{token}"',
        shell=True, capture_output=True, text=True
    )
    if res.returncode != 0:
        print(f"❌ Zrok Enable Failed:\n{res.stderr}")
        return False
    print("✅ Zrok Authenticated.")
    return True

def start_zrok_tunnel_loop(port):
    print(f"   Opening Zrok tunnel for port {port}...")
    share_proc = subprocess.Popen(
        [ZROK_BINARY, "share", "public", f"localhost:{port}", "--headless"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
    )
    for i in range(10):
        time.sleep(3)
        if share_proc.poll() is not None:
            print(f"❌ Zrok share process died.\nSTDERR: {share_proc.stderr.read()}")
            return "Error"
        res = subprocess.run([ZROK_BINARY, "overview"], capture_output=True, text=True)
        m = re.search(r'(https?://[a-z0-9\-\.]+\.zrok\.io)', res.stdout)
        if m: return m.group(1)
    return "Timeout"

# ---- 1. CLEANUP ----
print("🧹 Killing old processes...")
os.system(f"fuser -k {COMFY_PORT}/tcp")
os.system(f"fuser -k {FACADE_PORT}/tcp")
os.system("pkill -9 zrok")
os.system("pkill -f zimage_api_t2i.py")
time.sleep(2)

# ---- 2. START COMFYUI ----
print("🚀 Starting ComfyUI Server...")
comfy_proc = subprocess.Popen(
    [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", str(COMFY_PORT)],
    cwd=COMFY_DIR,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("⏳ Waiting for ComfyUI to be ready...")
comfy_ready = False
for i in range(90):
    try:
        if requests.get(f"http://127.0.0.1:{COMFY_PORT}/system_stats", timeout=2).status_code == 200:
            print(f"✅ ComfyUI is Ready (after {i+1}s).")
            comfy_ready = True
            break
    except: pass
    time.sleep(1)

if not comfy_ready:
    print("❌ ComfyUI failed to start within 90s. Check your setup.")
else:
    # ---- 3. START HEADLESS API ----
    print(f"🚀 Starting Z-Image API (zimage_api_t2i.py) on port {FACADE_PORT}...")
    api_proc = subprocess.Popen(
        [sys.executable, "zimage_api_t2i.py"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )
    time.sleep(3)

    try:
        r = requests.get(f"http://127.0.0.1:{FACADE_PORT}/", timeout=5)
        print(f"✅ Z-Image API ready: {r.json()}")
    except:
        print("⚠️ API may still be starting...")

    # ---- 4. START ZROK ----
    token = get_zrok_token()
    if not token:
        print("⚠️ Zrok token not found. Check dataset is attached.")
    elif force_zrok_auth(token):
        url = start_zrok_tunnel_loop(FACADE_PORT)
        print(f"""
========================================================
🎨 Z-IMAGE T2I API LIVE
   Model  : z_image_bf16.safetensors (6B S3-DiT)
   Encoder: qwen_3_4b.safetensors
   VAE    : ae.safetensors

🌍 Public URL : {url}

📡 Endpoints:
   GET  {url}/              → Health check
   POST {url}/generate-async → Queue generation
   GET  {url}/status/{{task_id}} → Poll & retrieve image

📋 Parameters (form data):
   prompt              (required)
   negative_prompt     (default: "")
   num_inference_steps (default: 30, recommended: 30-50)
   guidance_scale      (default: 4.0, recommended: 3-5)
   width               (default: 1024)
   height              (default: 1024)
   seed                (optional, random if omitted)
========================================================
""")

        # ---- 5. KEEPALIVE LOOP ----
        try:
            while True:
                if api_proc.poll() is not None:
                    print("❌ Z-Image API process died! Stderr:")
                    print(api_proc.stderr.read().decode())
                    break
                if comfy_proc.poll() is not None:
                    print("❌ ComfyUI process died!")
                    break
                time.sleep(60)
        except KeyboardInterrupt:
            print("\n🛑 Shutting down...")
            comfy_proc.terminate()
            api_proc.terminate()
            os.system("pkill zrok")
            print("✅ Shutdown complete.")
    else:
        print("⚠️ Zrok authentication failed. Check your token.")
